# Session 14 — Train a linear model, convert it to an MoE

This experiment trains a scalar linear regressor on a synthetic piecewise-linear task, converts its learned weights into three experts, verifies that conversion initially preserves every prediction, and then continues training. The held-out MSE must decrease or the script raises an error.

The synthetic target uses three input regions with different slopes. It is intentionally small and deterministic so the architecture change and continued learning are easy to inspect and reproduce. This is a controlled demonstration, not a claim about large-scale language-model MoE performance.

In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/ERA_Portfolio")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/dhruv369/ERA_Portfolio.git", str(repo)],
        check=True,
    )
%cd /content/ERA_Portfolio/Assignments/S14

## Run the experiment

The script prints the baseline and final train/validation MSE, parameter counts, number of training steps, and the maximum prediction difference at conversion. Results and the comparison/router plot are saved to Google Drive so they survive a Colab reset.

In [ ]:
from google.colab import drive
from IPython.display import Image, display
import json
import os
import subprocess

drive.mount("/content/drive")
os.environ["S14_OUTPUT_DIR"] = "/content/drive/MyDrive/ERA_S14/output"
subprocess.run(["python", "-u", "session14_moe.py"], check=True)

output_dir = Path(os.environ["S14_OUTPUT_DIR"])
results = json.loads((output_dir / "session14_results.json").read_text())
print("\nHeld-out MSE before/after:", results["linear_validation_mse"], "->", results["moe_final_validation_mse"])
display(Image(filename=str(output_dir / "linear_to_moe.png")))
display(Image(filename=str(output_dir / "session14_training_loss.png")))

## Interpret the result

Check that the conversion prediction error is near zero, then compare validation MSE before and after the MoE training phase. The router plot should show how input regions are assigned to experts. The exact numerical results are written to `session14_results.json`.